# Imports

In [ ]:
!pip install XlsxWriter

In [ ]:
import gc
import os
import json
import joblib
import warnings
import numpy as np
import pandas as pd
from scipy.ndimage import label
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score,
    roc_curve,
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    average_precision_score
)
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.preprocessing import QuantileTransformer
from imblearn.under_sampling import RandomUnderSampler
import xlsxwriter
print("xlsxwriter version:", xlsxwriter.__version__)
import glob
import re
import matplotlib.pyplot as plt
from sklearn.metrics import auc
import seaborn as sns
from scipy.stats import mannwhitneyu
from IPython.display import display

plt.style.use("seaborn-v0_8-whitegrid")
warnings.filterwarnings("ignore")


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# Setup & Globals

In [ ]:
basepath = "/content/drive/MyDrive/DATASIGNALPROCESSING"
dataset = f"{basepath}/Data"
inputpath = f"{dataset}/04_CHB_MIT_WPE_Features_final_std"
output_root = f"{basepath}/Output"
plot_root = f"{output_root}/Plots"
os.makedirs(plot_root, exist_ok=True)
os.makedirs(output_root, exist_ok=True)
ABLATION_ROOT = os.path.join(output_root, "ablation_study_methodology_coherent")
os.makedirs(ABLATION_ROOT, exist_ok=True)


In [ ]:
EPOCH_SEC = 2.5
FAST_WIN = 12                  # 30 s
SLOW_WIN = 480                 # ~20 min
SMOOTH_WIN = 240               # ~6.25 min
WARMUP_CUTOFF = 480            # ~20 min
PERSISTENCE_MIN = 10
PERSISTENCE_EPOCHS = int((PERSISTENCE_MIN * 60) / EPOCH_SEC)
UNDERSAMPLING_STRATEGY = 0.4
BLEND_ALPHA = 0.7              # RF weight

RF_PARAMS = dict(
    n_estimators=120,
    max_depth=5,
    min_samples_leaf=100,
    class_weight="balanced_subsample",
    n_jobs=-1,
    random_state=42
)

logo = LeaveOneGroupOut()

channel_names = [
    "FP1-F7","F7-T7","T7-P7","P7-O1",
    "FP1-F3","F3-C3","C3-P3","P3-O1",
    "FP2-F4","F4-C4","C4-P4","P4-O2",
    "FP2-F8","F8-T8","T8-P8","P8-O2",
    "FZ-CZ","CZ-PZ"
]

In [ ]:
raw_feature_names = [f"Ch_{i+1:02d}" for i in range(18)]
delta_feature_names = [f"Delta_{i+1:02d}" for i in range(18)]
vol_feature_names = [f"Volatility_{i+1:02d}" for i in range(18)]
sync_feature_names = ["Synchrony"]

full_feature_names = (
    raw_feature_names +
    delta_feature_names +
    vol_feature_names +
    sync_feature_names
)

# Dataset

In [ ]:
X_all = np.load(f"{inputpath}/X_baseline_std.npy").astype(np.float32)
y_all = np.load(f"{inputpath}/y_baseline_std.npy").astype(np.int32)
g_all = np.load(f"{inputpath}/groups_baseline_std.npy").astype(np.int32)

print("X_all:", X_all.shape)
print("y_all:", y_all.shape)
print("g_all:", g_all.shape)

raw_directions = np.load(f"{inputpath}/wpe_directions.npy", allow_pickle=True)
dir_dict = raw_directions.item() if isinstance(raw_directions, np.ndarray) and raw_directions.dtype == object else raw_directions
patients = np.unique(g_all)
dir_map = {int(pid): int(dir_dict.get(pid, 1)) for pid in patients}

# Utils & Functions

In [ ]:
def make_config_dirs(config_key):
    root = os.path.join(ABLATION_ROOT, config_key)
    dirs = {
        "root": root,
        "models": os.path.join(root, "models"),
        "metrics": os.path.join(root, "metrics"),
        "predictions": os.path.join(root, "predictions"),
        "plots_data": os.path.join(root, "plots_data"),
        "config": os.path.join(root, "config_info.json"),
    }
    for k, v in dirs.items():
        if k != "config":
            os.makedirs(v, exist_ok=True)
    return dirs


def save_config_info(dirs, config_key, n_features, use_persistence):
    payload = {
        "config_key": config_key,
        "n_features": int(n_features),
        "epoch_sec": EPOCH_SEC,
        "fast_window_epochs": FAST_WIN,
        "slow_window_epochs": SLOW_WIN,
        "smoothing_window_epochs": SMOOTH_WIN,
        "warmup_cutoff_epochs": WARMUP_CUTOFF,
        "persistence_enabled": bool(use_persistence),
        "persistence_minutes": PERSISTENCE_MIN if use_persistence else 0,
        "persistence_epochs": PERSISTENCE_EPOCHS if use_persistence else 0,
        "undersampling_strategy": UNDERSAMPLING_STRATEGY,
        "rf_params": RF_PARAMS,
        "normalization": "patient-wise quantile normalization after polarity correction",
        "validation": "Leave-One-Patient-Out",
        "note": "Methodology-coherent ablation study"
    }
    with open(dirs["config"], "w") as f:
        json.dump(payload, f, indent=2)

In [ ]:
# SIGNAL / FEATURES
def smooth_probabilities(y_prob, win=SMOOTH_WIN):
    return (
        pd.Series(y_prob)
        .rolling(window=win, center=True, min_periods=1)
        .mean()
        .ffill()
        .bfill()
        .values
    )

def apply_persistence_rule(y_pred_binary, n_epochs=PERSISTENCE_EPOCHS):
    return (
        pd.Series(y_pred_binary)
        .rolling(window=n_epochs, min_periods=n_epochs)
        .min()
        .fillna(0)
        .astype(int)
        .values
    )

def get_false_alarm_metrics(y_true, y_pred_binary, epoch_sec=EPOCH_SEC):
    alarm_clusters, num_alarms = label(y_pred_binary)
    false_alarm_events = 0

    for alarm_id in range(1, num_alarms + 1):
        cluster_mask = (alarm_clusters == alarm_id)
        if np.sum(y_true[cluster_mask]) == 0:
            false_alarm_events += 1

    interictal_hours = (np.sum(y_true == 0) * epoch_sec) / 3600.0
    far_h = false_alarm_events / interictal_hours if interictal_hours > 0 else 0.0
    return far_h, false_alarm_events, interictal_hours

def compute_event_metrics(y_true, y_pred, epoch_sec=EPOCH_SEC, sph_min=5):
    preictal_clusters, num_seizures = label(y_true)

    predicted_seizures = 0
    warning_times = []

    for i in range(1, num_seizures + 1):
        cluster_mask = (preictal_clusters == i)
        cluster_indices = np.where(cluster_mask)[0]
        alarms_in_cluster = y_pred[cluster_mask]

        if np.any(alarms_in_cluster):
            predicted_seizures += 1
            first_alarm_relative_idx = np.where(alarms_in_cluster)[0][0]
            first_alarm_global_idx = cluster_indices[first_alarm_relative_idx]
            end_of_preictal_idx = cluster_indices[-1]

            time_to_end_of_preictal_sec = (end_of_preictal_idx - first_alarm_global_idx) * epoch_sec
            warning_time_sec = time_to_end_of_preictal_sec + (sph_min * 60)
            warning_times.append(warning_time_sec / 60.0)

    missed_seizures = num_seizures - predicted_seizures
    seizure_sensitivity = (predicted_seizures / num_seizures) if num_seizures > 0 else np.nan

    mean_warning_time = float(np.mean(warning_times)) if warning_times else np.nan
    median_warning_time = float(np.median(warning_times)) if warning_times else np.nan

    alarm_clusters, num_alarms = label(y_pred)
    false_alarm_events = 0
    for i in range(1, num_alarms + 1):
        cluster_mask = (alarm_clusters == i)
        if np.sum(y_true[cluster_mask]) == 0:
            false_alarm_events += 1

    interictal_epochs = np.sum(y_true == 0)
    interictal_hours = (interictal_epochs * epoch_sec) / 3600.0
    far_h = false_alarm_events / interictal_hours if interictal_hours > 0 else 0.0
    far_24h = far_h * 24.0

    interictal_alarms = np.sum((y_pred == 1) & (y_true == 0))
    time_in_warning = (interictal_alarms / interictal_epochs) if interictal_epochs > 0 else 0.0

    return {
        "total_seizures": int(num_seizures),
        "predicted_seizures": int(predicted_seizures),
        "missed_seizures": int(missed_seizures),
        "seizure_sensitivity": seizure_sensitivity,
        "mean_warning_time_min": mean_warning_time,
        "median_warning_time_min": median_warning_time,
        "false_alarm_events": int(false_alarm_events),
        "far_h": far_h,
        "far_24h": far_24h,
        "interictal_hours": float(interictal_hours),
        "time_in_warning": time_in_warning
    }

def extract_raw_wpe(X_patient):
    return X_patient[:, :18].astype(np.float32)


def extract_calibrated_wpe(X_patient, patient_id):
    X_raw = extract_raw_wpe(X_patient)
    direction = dir_map.get(int(patient_id), 1)
    X_corrected = X_raw * direction
    return X_corrected.astype(np.float32)


def extract_univariate_score(X_patient, patient_id):
    X_base = extract_calibrated_wpe(X_patient, patient_id)
    return X_base.mean(axis=1).astype(np.float32)


def extract_full_feature_space(X_patient, patient_id):
    X_base = extract_calibrated_wpe(X_patient, patient_id)
    df = pd.DataFrame(X_base)

    ma_fast = df.rolling(window=FAST_WIN, min_periods=1).mean()
    ma_slow = df.rolling(window=SLOW_WIN, min_periods=1).mean()
    X_delta = (ma_fast - ma_slow).values.astype(np.float32)

    X_volatility = (
        df.rolling(window=FAST_WIN, min_periods=1)
        .std()
        .fillna(0.0)
        .values
        .astype(np.float32)
    )

    X_sync = (
        df.std(axis=1)
        .rolling(window=FAST_WIN, min_periods=1)
        .mean()
        .fillna(0.0)
        .values
        .reshape(-1, 1)
        .astype(np.float32)
    )

    X_full = np.hstack([X_base, X_delta, X_volatility, X_sync]).astype(np.float32)
    return X_full


def get_features_for_config(config_key, X_patient, patient_id):
    if config_key == "A_raw_wpe":
        return extract_raw_wpe(X_patient)

    if config_key == "B_wpe_calibration":
        return extract_calibrated_wpe(X_patient, patient_id)

    if config_key in ["C_full_feature_space", "D_full_feature_space_persistence"]:
        return extract_full_feature_space(X_patient, patient_id)

    raise ValueError(f"Unknown config_key: {config_key}")


def summarize_pooled_seizure_counts(df_metrics, out_csv_path):
    total = int(df_metrics["total_seizures"].sum())
    predicted = int(df_metrics["predicted_seizures"].sum())
    missed = int(df_metrics["missed_seizures"].sum())
    pooled_sensitivity = (predicted / total) if total > 0 else np.nan

    df_pooled = pd.DataFrame([{
        "total_seizures": total,
        "predicted_seizures": predicted,
        "missed_seizures": missed,
        "pooled_seizure_sensitivity": pooled_sensitivity
    }])
    df_pooled.to_csv(out_csv_path, index=False)
    return df_pooled


In [ ]:
#Metrics
def choose_threshold_youden(y_true, y_score):
    fpr, tpr, thresholds = roc_curve(y_true, y_score)
    best_idx = np.argmax(tpr - fpr)
    return thresholds[best_idx], fpr, tpr, thresholds

def choose_threshold_youden_nested(train_patients, X_train_blocks, y_train_blocks,
                                    univariate_train_blocks, rf_params, blend_alpha,
                                    undersampling_strategy, n_splits=4,
                                    warmup=WARMUP_CUTOFF, smooth_win=SMOOTH_WIN,
                                    random_state=42):
    rng = np.random.RandomState(random_state)
    shuffled = list(train_patients)
    rng.shuffle(shuffled)
    groups = np.array_split(shuffled, min(n_splits, len(shuffled)))

    idx_by_patient = {p: i for i, p in enumerate(train_patients)}

    oof_y, oof_score = [], []

    for group in groups:
        group = list(group)
        inner_train_patients = [p for p in train_patients if p not in group]
        if len(inner_train_patients) == 0 or len(group) == 0:
            continue

        inner_idx = [idx_by_patient[p] for p in inner_train_patients]
        X_inner_train = np.vstack([X_train_blocks[i] for i in inner_idx]).astype(np.float32)
        y_inner_train = np.concatenate([y_train_blocks[i] for i in inner_idx]).astype(np.int32)
        uni_inner_train = np.concatenate([univariate_train_blocks[i] for i in inner_idx]).astype(np.float32)

        rus = RandomUnderSampler(sampling_strategy=undersampling_strategy, random_state=random_state)
        X_res, y_res = rus.fit_resample(X_inner_train, y_inner_train)

        clf_inner = RandomForestClassifier(**rf_params)
        clf_inner.fit(X_res, y_res)

        logreg_inner = LogisticRegression(class_weight="balanced", random_state=random_state)
        logreg_inner.fit(uni_inner_train.reshape(-1, 1), y_inner_train)

        for p in group:
            i = idx_by_patient[p]
            X_p = X_train_blocks[i]
            y_p = y_train_blocks[i]
            uni_p = univariate_train_blocks[i]

            if len(y_p) <= warmup or len(np.unique(y_p[warmup:])) < 2:
                continue

            prob_rf = clf_inner.predict_proba(X_p)[:, 1]
            prob_uni = logreg_inner.predict_proba(uni_p.reshape(-1, 1))[:, 1]
            prob_raw = (blend_alpha * prob_rf + (1 - blend_alpha) * prob_uni).astype(np.float32)
            prob_smooth = smooth_probabilities(prob_raw, win=smooth_win)

            oof_y.append(y_p[warmup:])
            oof_score.append(prob_smooth[warmup:])

        del clf_inner, X_res, y_res
        gc.collect()

    if len(oof_y) == 0:
        return None

    oof_y = np.concatenate(oof_y)
    oof_score = np.concatenate(oof_score)
    threshold_opt, _, _, _ = choose_threshold_youden(oof_y, oof_score)
    return threshold_opt


def compute_epoch_metrics(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="binary", zero_division=0
    )

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    fpr_epoch = fp / (fp + tn) if (fp + tn) > 0 else 0.0

    return {
        "accuracy": acc,
        "precision": prec,
        "sensitivity": rec,
        "specificity": spec,
        "f1_score": f1,
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
        "fpr_epoch": fpr_epoch
    }


def summarize_metrics(df_metrics, out_csv_path):
    metric_cols = [
        "auc_roc",
        "pr_auc",
        "accuracy",
        "precision",
        "sensitivity",
        "specificity",
        "f1_score",
        "far_h",
        "fpr_epoch",
        "false_alarm_events",
        "interictal_hours",
        "threshold"
    ]

    rows = []
    for col in metric_cols:
        rows.append({
            "Metric": col,
            "Mean": df_metrics[col].mean(),
            "Std": df_metrics[col].std(),
            "Min": df_metrics[col].min(),
            "Max": df_metrics[col].max(),
            "Median": df_metrics[col].median()
        })

    df_summary = pd.DataFrame(rows)
    df_summary.to_csv(out_csv_path, index=False)
    return df_summary


In [ ]:
from sklearn.linear_model import LogisticRegression


def run_ablation_config(config_key, use_persistence=False):
    dirs = make_config_dirs(config_key)

    all_metrics = []
    all_roc_data = []
    all_importances = []

    feature_names = (
        raw_feature_names if config_key == "A_raw_wpe"
        else raw_feature_names if config_key == "B_wpe_calibration"
        else full_feature_names
    )

    save_config_info(
        dirs=dirs,
        config_key=config_key,
        n_features=len(feature_names),
        use_persistence=use_persistence
    )

    for fold_idx, (train_idx, test_idx) in enumerate(logo.split(X_all, y_all, g_all), start=1):
        patient_test = int(g_all[test_idx][0])

        X_test_raw = X_all[test_idx]
        y_test_full = y_all[test_idx]

        X_test = get_features_for_config(config_key, X_test_raw, patient_test)
        univariate_test = extract_univariate_score(X_test_raw, patient_test)

        X_train_blocks = []
        y_train_blocks = []
        univariate_train_blocks = []

        train_patients = [p for p in patients if p != patient_test]

        for patient_train in train_patients:
            patient_mask = (g_all == patient_train)
            X_train_patient_raw = X_all[patient_mask]
            y_train_patient = y_all[patient_mask]

            X_train_patient = get_features_for_config(
                config_key,
                X_train_patient_raw,
                patient_train
            )
            univariate_train_patient = extract_univariate_score(X_train_patient_raw, patient_train)

            X_train_blocks.append(X_train_patient)
            y_train_blocks.append(y_train_patient)
            univariate_train_blocks.append(univariate_train_patient)

        X_train = np.vstack(X_train_blocks).astype(np.float32)
        y_train = np.concatenate(y_train_blocks).astype(np.int32)
        univariate_train = np.concatenate(univariate_train_blocks).astype(np.float32)

        rus = RandomUnderSampler(
            sampling_strategy=UNDERSAMPLING_STRATEGY,
            random_state=42
        )
        X_res, y_res = rus.fit_resample(X_train, y_train)

        clf = RandomForestClassifier(**RF_PARAMS)
        clf.fit(X_res, y_res)

        logreg_univariate = LogisticRegression(class_weight="balanced", random_state=42)
        logreg_univariate.fit(univariate_train.reshape(-1, 1), y_train)

        model_path = os.path.join(dirs["models"], f"rf_model_chb{patient_test:02d}.joblib")
        joblib.dump(clf, model_path)

        y_prob_rf = clf.predict_proba(X_test)[:, 1]
        y_prob_univariate = logreg_univariate.predict_proba(univariate_test.reshape(-1, 1))[:, 1]
        y_prob_raw = (BLEND_ALPHA * y_prob_rf + (1 - BLEND_ALPHA) * y_prob_univariate).astype(np.float32)
        y_prob_smooth = smooth_probabilities(y_prob_raw, win=SMOOTH_WIN)

        y_eval = y_test_full[WARMUP_CUTOFF:]
        y_score_eval = y_prob_smooth[WARMUP_CUTOFF:]
        y_prob_raw_eval = y_prob_raw[WARMUP_CUTOFF:]

        if len(np.unique(y_eval)) < 2:
            print(f"[{config_key}] chb{patient_test:02d} skipped (single-class after warm-up).")
            continue

        auc_roc = roc_auc_score(y_eval, y_score_eval)
        pr_auc = average_precision_score(y_eval, y_score_eval)

        threshold_opt_nested = choose_threshold_youden_nested(
            train_patients, X_train_blocks, y_train_blocks, univariate_train_blocks,
            RF_PARAMS, BLEND_ALPHA, UNDERSAMPLING_STRATEGY, n_splits=2,
            warmup=WARMUP_CUTOFF, smooth_win=SMOOTH_WIN, random_state=42
        )
        _, fpr, tpr, thresholds = choose_threshold_youden(y_eval, y_score_eval)
        threshold_opt = (
            threshold_opt_nested if threshold_opt_nested is not None
            else choose_threshold_youden(y_eval, y_score_eval)[0] )

        y_pred_binary = (y_score_eval >= threshold_opt).astype(int)
        y_pred_final = (
            apply_persistence_rule(y_pred_binary, PERSISTENCE_EPOCHS)
            if use_persistence else y_pred_binary)

        epoch_metrics = compute_epoch_metrics(y_eval, y_pred_final)
        event_metrics = compute_event_metrics(
            y_eval, y_pred_final, epoch_sec=EPOCH_SEC, sph_min=5
        )
        far_h = event_metrics["far_h"]
        false_alarm_events = event_metrics["false_alarm_events"]
        interictal_hours = event_metrics["interictal_hours"]

        patient_metrics = {
            "patient": patient_test,
            "config": config_key,
            "n_epochs_test_total": int(len(y_test_full)),
            "n_epochs_eval": int(len(y_eval)),
            "n_features": int(X_test.shape[1]),
            "auc_roc": auc_roc,
            "pr_auc": pr_auc,
            "threshold": float(threshold_opt),
            "accuracy": epoch_metrics["accuracy"],
            "precision": epoch_metrics["precision"],
            "sensitivity": epoch_metrics["sensitivity"],
            "specificity": epoch_metrics["specificity"],
            "f1_score": epoch_metrics["f1_score"],
            "tn": epoch_metrics["tn"],
            "fp": epoch_metrics["fp"],
            "fn": epoch_metrics["fn"],
            "tp": epoch_metrics["tp"],
            "fpr_epoch": epoch_metrics["fpr_epoch"],
            "far_h": far_h,
            "false_alarm_events": int(false_alarm_events),
            "interictal_hours": float(interictal_hours),
            "total_seizures": event_metrics["total_seizures"],
            "predicted_seizures": event_metrics["predicted_seizures"],
            "missed_seizures": event_metrics["missed_seizures"],
            "seizure_sensitivity": event_metrics["seizure_sensitivity"],
            "mean_warning_time_min": event_metrics["mean_warning_time_min"],
            "median_warning_time_min": event_metrics["median_warning_time_min"],
            "far_24h": event_metrics["far_24h"],
            "time_in_warning": event_metrics["time_in_warning"]
        }
        all_metrics.append(patient_metrics)

        pred_df = pd.DataFrame({
            "epoch_index_global": np.arange(len(y_test_full)),
            "epoch_index_eval": np.concatenate([
                np.full(WARMUP_CUTOFF, np.nan),
                np.arange(len(y_eval))
            ]),
            "time_minutes_global": np.arange(len(y_test_full)) * EPOCH_SEC / 60.0,
            "y_true_full": y_test_full,
            "y_prob_raw_full": y_prob_raw,
            "y_prob_smooth_full": y_prob_smooth,
        })

        pred_df["is_eval_epoch"] = 0
        pred_df.loc[WARMUP_CUTOFF:, "is_eval_epoch"] = 1

        pred_df["threshold_eval"] = np.nan
        pred_df.loc[WARMUP_CUTOFF:, "threshold_eval"] = threshold_opt

        pred_df["y_pred_binary_eval"] = np.nan
        pred_df["y_pred_final_eval"] = np.nan
        pred_df.loc[WARMUP_CUTOFF:, "y_pred_binary_eval"] = y_pred_binary
        pred_df.loc[WARMUP_CUTOFF:, "y_pred_final_eval"] = y_pred_final

        pred_path = os.path.join(
            dirs["predictions"],
            f"patient_chb{patient_test:02d}_predictions.csv"
        )
        pred_df.to_csv(pred_path, index=False)

        roc_df = pd.DataFrame({
            "fpr": fpr,
            "tpr": tpr,
            "thresholds": thresholds
        })
        roc_path = os.path.join(
            dirs["plots_data"],
            f"patient_chb{patient_test:02d}_roc.csv"
        )
        roc_df.to_csv(roc_path, index=False)

        all_roc_data.append({
            "patient": patient_test,
            "fpr": fpr,
            "tpr": tpr,
            "thresholds": thresholds,
            "auc_roc": auc_roc
        })

        if hasattr(clf, "feature_importances_"):
            imp_df = pd.DataFrame({
                "feature": feature_names,
                "importance": clf.feature_importances_
            }).sort_values("importance", ascending=False)

            imp_path = os.path.join(
                dirs["plots_data"],
                f"patient_chb{patient_test:02d}_feature_importance.csv"
            )
            imp_df.to_csv(imp_path, index=False)

            all_importances.append({
                "patient": patient_test,
                "importance": clf.feature_importances_
            })

        print(
            f"[{config_key}] Fold {fold_idx:02d} | chb{patient_test:02d} | "
            f"AUC={auc_roc:.3f} | Sens={patient_metrics['sensitivity']:.3f} | "
            f"Spec={patient_metrics['specificity']:.3f} | FAR/h={far_h:.3f} | "
            f"Seizures {event_metrics['predicted_seizures']}/{event_metrics['total_seizures']} | "
            f"MedianWarn={event_metrics['median_warning_time_min']:.1f}min | "
            f"FAR/24h={event_metrics['far_24h']:.2f}"
        )

        del X_train, y_train, X_res, y_res, X_test, y_prob_raw, y_prob_smooth, clf
        gc.collect()

    df_metrics = pd.DataFrame(all_metrics)

    subject_metrics_path = os.path.join(dirs["metrics"], "subject_metrics.csv")
    df_metrics.to_csv(subject_metrics_path, index=False)

    summary_path = os.path.join(dirs["metrics"], "global_metrics.csv")
    df_summary = summarize_metrics(df_metrics, summary_path)

    pooled_seizure_path = os.path.join(dirs["metrics"], "pooled_seizure_counts.csv")
    df_pooled_seizures = summarize_pooled_seizure_counts(df_metrics, pooled_seizure_path)

    excel_path = os.path.join(dirs["metrics"], "metrics.xlsx")
    with pd.ExcelWriter(excel_path, engine="xlsxwriter") as writer:
        df_summary.to_excel(writer, sheet_name="GlobalMetrics", index=False)
        df_metrics.to_excel(writer, sheet_name="SubjectMetrics", index=False)
        df_pooled_seizures.to_excel(writer, sheet_name="PooledSeizureCounts", index=False)

    if len(all_importances) > 0:
        importances_matrix = np.vstack([x["importance"] for x in all_importances])
        mean_importance = importances_matrix.mean(axis=0)
        std_importance = importances_matrix.std(axis=0)

        df_importance_global = pd.DataFrame({
            "feature": feature_names,
            "mean_importance": mean_importance,
            "std_importance": std_importance
        }).sort_values("mean_importance", ascending=False)

        df_importance_global.to_csv(
            os.path.join(dirs["plots_data"], "global_feature_importance.csv"),
            index=False
        )

    bundle = {
        "config_key": config_key,
        "feature_names": feature_names,
        "all_metrics": all_metrics,
        "all_roc_data": all_roc_data,
        "all_importances": all_importances
    }
    joblib.dump(bundle, os.path.join(dirs["plots_data"], "results_bundle.joblib"))

    print(f"\nCompleted: {config_key}")
    print(df_summary)
    print("\nPooled seizure-level counts (cohort-wide, not per-patient average):")
    print(df_pooled_seizures)

    return df_metrics, df_summary


# LOPO CROSS VALIDATION + Ablation

In [ ]:
df_A, summary_A = run_ablation_config(
    config_key="A_raw_wpe",
    use_persistence=False
)

In [ ]:
df_B, summary_B = run_ablation_config(
    config_key="B_wpe_calibration",
    use_persistence=False
)

In [ ]:
df_C, summary_C = run_ablation_config(
    config_key="C_full_feature_space",
    use_persistence=False
)

In [ ]:
df_D, summary_D = run_ablation_config(
    config_key="D_full_feature_space_persistence",
    use_persistence=True
)

In [ ]:
def load_global_metric_mean(config_dir):
    df = pd.read_csv(os.path.join(config_dir, "metrics", "global_metrics.csv"))
    out = {}
    for _, row in df.iterrows():
        out[row["Metric"]] = {
            "Mean": row["Mean"],
            "Std": row["Std"]
        }
    return out


def safe_stat(stats, key, field, default=float("nan")):
    """Ritorna Mean/Std se la metrica esiste nel csv, altrimenti NaN
    (utile se A/B/C sono state calcolate prima di aggiungere le metriche event-level)."""
    return stats.get(key, {}).get(field, default)


combined_rows = []
config_map = {
    "A_raw_wpe": os.path.join(ABLATION_ROOT, "A_raw_wpe"),
    "B_wpe_calibration": os.path.join(ABLATION_ROOT, "B_wpe_calibration"),
    "C_full_feature_space": os.path.join(ABLATION_ROOT, "C_full_feature_space"),
    "D_full_feature_space_persistence": os.path.join(ABLATION_ROOT, "D_full_feature_space_persistence"),
}

for config_key, config_dir in config_map.items():
    stats = load_global_metric_mean(config_dir)
    combined_rows.append({
        "Config": config_key,
        "AUC-ROC Mean": safe_stat(stats, "auc_roc", "Mean"),
        "AUC-ROC Std": safe_stat(stats, "auc_roc", "Std"),
        "PR-AUC Mean": safe_stat(stats, "pr_auc", "Mean"),
        "PR-AUC Std": safe_stat(stats, "pr_auc", "Std"),
        "Sensitivity Mean": safe_stat(stats, "sensitivity", "Mean"),
        "Sensitivity Std": safe_stat(stats, "sensitivity", "Std"),
        "Specificity Mean": safe_stat(stats, "specificity", "Mean"),
        "Specificity Std": safe_stat(stats, "specificity", "Std"),
        "Precision Mean": safe_stat(stats, "precision", "Mean"),
        "Precision Std": safe_stat(stats, "precision", "Std"),
        "F1 Mean": safe_stat(stats, "f1_score", "Mean"),
        "F1 Std": safe_stat(stats, "f1_score", "Std"),
        "FAR/h Mean": safe_stat(stats, "far_h", "Mean"),
        "FAR/h Std": safe_stat(stats, "far_h", "Std"),
        "Seizure Sensitivity Mean": safe_stat(stats, "seizure_sensitivity", "Mean"),
        "Seizure Sensitivity Std": safe_stat(stats, "seizure_sensitivity", "Std"),
        "Mean Warning Time (min)": safe_stat(stats, "mean_warning_time_min", "Mean"),
        "Median Warning Time (min)": safe_stat(stats, "median_warning_time_min", "Mean"),
        "FAR/24h Mean": safe_stat(stats, "far_24h", "Mean"),
        "FAR/24h Std": safe_stat(stats, "far_24h", "Std"),
        "Time-in-Warning Mean": safe_stat(stats, "time_in_warning", "Mean"),
    })

df_combined = pd.DataFrame(combined_rows)
combined_path = os.path.join(ABLATION_ROOT, "ablation_summary_all_configs.csv")
df_combined.to_csv(combined_path, index=False)

print("\nAll configurations completed.")
display(df_combined)

# Plots

In [ ]:
config_dir = os.path.join(ABLATION_ROOT, "D_full_feature_space_persistence")

roc_dir = os.path.join(config_dir, "plots_data")
roc_files = sorted(glob.glob(os.path.join(roc_dir, "patient_chb*_roc.csv")))

mean_fpr = np.linspace(0, 1, 100)
tprs = []
aucs = []
per_patient_auc = []

plt.rcParams.update({
    "font.size": 12,
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "legend.fontsize": 9,
    "figure.dpi": 120
})

plt.figure(figsize=(8.5, 7.5))

for roc_file in roc_files:
    match = re.search(r"chb(\d+)", os.path.basename(roc_file))
    if match is None:
        continue
    pid = int(match.group(1))
    df_roc = pd.read_csv(roc_file)
    fpr = df_roc["fpr"].values
    tpr = df_roc["tpr"].values
    # AUC computed from ROC file
    auc_val = auc(fpr, tpr)
    # interpolation for AUc average
    interp_tpr = np.interp(mean_fpr, fpr, tpr)
    interp_tpr[0] = 0.0
    tprs.append(interp_tpr)
    aucs.append(auc_val)
    per_patient_auc.append({"patient": pid, "auc_roc": auc_val})
    plt.plot(fpr, tpr, lw=1.0, alpha=0.22, color="#7aa6d8")

if len(tprs) == 0:
    raise ValueError(f"No ROC CSV files found in {roc_dir}")

mean_tpr = np.mean(tprs, axis=0)
mean_tpr[-1] = 1.0
mean_auc = auc(mean_fpr, mean_tpr)
std_auc = np.std(aucs)
std_tpr = np.std(tprs, axis=0)
tprs_upper = np.minimum(mean_tpr + std_tpr, 1)
tprs_lower = np.maximum(mean_tpr - std_tpr, 0)

# ROC average
plt.plot(
    mean_fpr, mean_tpr,
    color="#0b3c5d",
    lw=2.8,
    label=f"Mean ROC (AUC = {mean_auc:.3f} ± {std_auc:.3f})"
)
# variability
plt.fill_between(
    mean_fpr, tprs_lower, tprs_upper,
    color="grey", alpha=0.20,
    label="± 1 std. dev.")
# Chance line
plt.plot(
    [0, 1], [0, 1],
    linestyle="--", color="#c0392b", lw=1.5, alpha=0.8, label="Chance"
)
plt.xlabel("FPR")
plt.ylabel("TPR")
plt.title("ROC Curve - LOPO", fontweight="bold")
plt.grid(alpha=0.25, linestyle="--")
plt.legend(loc="lower right")
plt.tight_layout()
save_path = os.path.join(roc_dir, "mean_roc_model_D.png")
plt.savefig(save_path, dpi=300, bbox_inches="tight")
plt.show()
df_auc = pd.DataFrame(per_patient_auc).sort_values("patient")
print(df_auc.to_string(index=False))
print(f"Saved ROC figure in: {save_path}")

In [ ]:
Xall = np.load(f"{inputpath}/X_baseline_std.npy").astype(np.float32)
yall = np.load(f"{inputpath}/y_baseline_std.npy").astype(np.int32)
gall = np.load(f"{inputpath}/groups_baseline_std.npy").astype(np.int32)

print("Input loaded")
print("Xall:", Xall.shape)
print("yall:", yall.shape)
print("gall:", gall.shape)

In [ ]:
def plot_patient_entropy_probability(pid, config_dir, Xall, yall, gall, save=True):
    pid_num = int(pid.replace("chb", ""))
    pred_path = os.path.join(config_dir, "predictions", f"patient_{pid}_predictions.csv")
    df_pred = pd.read_csv(pred_path)
    X_patient = Xall[gall == pid_num]
    y_patient = yall[gall == pid_num]
    wpe_signal = extract_calibrated_wpe(X_patient, pid_num).mean(axis=1)
    wpe_smooth = (
        pd.Series(wpe_signal)
        .rolling(window=150, center=True, min_periods=1)
        .mean()
        .values
    )
    y_prob_smooth = df_pred["y_prob_smooth_full"].values
    y_true = df_pred["y_true_full"].values
    threshold_vals = df_pred["threshold_eval"].dropna().values
    threshold = threshold_vals[0] if len(threshold_vals) > 0 else 0.5
    y_alarm = df_pred["y_pred_final_eval"].fillna(0).values
    time_axis = np.arange(len(y_true)) * 2.5 / 60.0  # min
    fig, (ax1, ax2) = plt.subplots(
        2, 1, figsize=(14, 8), sharex=True,
        gridspec_kw={"hspace": 0.10}
    )
    ax1.plot(time_axis, wpe_smooth, color="#c0392b", lw=2, label="Mean WPE Complexity Metric")
    ax1.fill_between(
        time_axis,
        np.nanmin(wpe_smooth),
        np.nanmax(wpe_smooth),
        where=(y_true == 1),
        color="#e74c3c",
        alpha=0.15,
        label="Pre-ictal Zone"
    )
    ax1.set_ylabel("Entropy WPE", fontweight="bold")
    ax1.set_title(f"Complexity Collapse Analysis - Patient {pid}", fontsize=16, fontweight="bold")
    ax1.grid(alpha=0.2, linestyle="--")
    ax1.legend(loc="upper right")
    ax2.plot(time_axis, y_prob_smooth, color="#2980b9", lw=2, label="Model Seizure Probability")
    ax2.fill_between(
        time_axis,
        0,
        y_alarm * 0.2,
        color="orange",
        alpha=0.8,
        label="Alarm Trigger (persistence)"
    )
    ax2.axhline(
        y=threshold,
        color="black",
        linestyle="--",
        alpha=0.6,
        label=f"Optimal Threshold = {threshold:.2f}"
    )
    ax2.fill_between(
        time_axis,
        0,
        1,
        where=(y_true == 1),
        color="#e74c3c",
        alpha=0.10
    )
    ax2.set_ylabel("Probability", fontweight="bold")
    ax2.set_xlabel("Time (minutes)", fontweight="bold")
    ax2.set_ylim(-0.05, 1.05)
    ax2.grid(alpha=0.2, linestyle="--")
    ax2.legend(loc="upper right")
    plt.tight_layout()
    if save:
        save_dir = os.path.join(config_dir, "plots_data", "patient_entropy_probability")
        os.makedirs(save_dir, exist_ok=True)
        save_path = os.path.join(save_dir, f"complexity_probability_{pid}.png")
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
        print(f"Saved in: {save_path}")

    plt.show()

In [ ]:
pid = 1
pred_path = os.path.join(config_dir, "predictions", f"patient_chb{int(pid):02d}_predictions.csv")
df_pred = pd.read_csv(pred_path)
print(pred_path)
print(df_pred.columns.tolist())

In [ ]:
plot_patient_entropy_probability(
    pid="chb01",
    config_dir=config_dir,
    Xall=X_all,
    yall=y_all,
    gall=g_all,
    save=True
)

In [ ]:
patient_ids = sorted(df_D["patient"].unique())
for pid in patient_ids:
    try:
        plot_patient_entropy_probability(
            pid=pid,
            config_dir=config_dir,
            Xall=Xall,
            yall=yall,
            gall=gall,
            save=True
        )
    except Exception as e:
        print(f"{pid} failed: {e}")

In [ ]:

print(df_pred.columns.tolist())
print(df_pred.head())


## Features

In [ ]:
config_dir = os.path.join(ABLATION_ROOT, "D_full_feature_space_persistence")
imp_path = os.path.join(config_dir, "plots_data", "global_feature_importance.csv")
df_imp = pd.read_csv(imp_path)
display(df_imp.head(10))

In [ ]:
def feature_family(name):
    if name.startswith("Chi"):
        return "Calibrated WPE"
    elif name.startswith("Delta"):
        return "Temporal Trend"
    elif name.startswith("Volatility"):
        return "Local Volatility"
    elif name == "Synchrony":
        return "Spatial Synchrony"
    return "Other"

df_imp["family"] = df_imp["feature"].apply(feature_family)

df_family = (
    df_imp.groupby("family", as_index=False)["mean_importance"]
    .sum()
    .sort_values("mean_importance", ascending=False)
)
display(df_family)

In [ ]:
def feature_channel(name):
    if name == "Synchrony":
        return "Global"
    idx = int(name[-2:]) - 1
    return channel_names[idx]

df_imp["channel"] = df_imp["feature"].apply(feature_channel)
df_channel = (
    df_imp.groupby("channel", as_index=False)["mean_importance"]
    .sum()
    .sort_values("mean_importance", ascending=False)
)

display(df_channel)

In [ ]:
df_top10 = df_imp.copy()
df_top10["family"] = df_top10["feature"].apply(feature_family)
df_top10["channel"] = df_top10["feature"].apply(feature_channel)
df_top10 = df_top10.sort_values("mean_importance", ascending=False).head(10)
display(df_top10)

In [ ]:
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({
    "figure.dpi": 130,
    "axes.titlesize": 15,
    "axes.labelsize": 13,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "legend.fontsize": 10
})

CONFIG_MAP = {
    "A_raw_wpe": os.path.join(ABLATION_ROOT, "A_raw_wpe"),
    "B_wpe_calibration": os.path.join(ABLATION_ROOT, "B_wpe_calibration"),
    "C_full_feature_space": os.path.join(ABLATION_ROOT, "C_full_feature_space"),
    "D_full_feature_space_persistence": os.path.join(ABLATION_ROOT, "D_full_feature_space_persistence"),
}

FINAL_CONFIG = CONFIG_MAP["D_full_feature_space_persistence"]
PLOTS_OUT = os.path.join(FINAL_CONFIG, "plots_data", "feature_analysis")
os.makedirs(PLOTS_OUT, exist_ok=True)

In [ ]:
# Congfigurations
channel_names = [
    "FP1-F7","F7-T7","T7-P7","P7-O1",
    "FP1-F3","F3-C3","C3-P3","P3-O1",
    "FP2-F4","F4-C4","C4-P4","P4-O2",
    "FP2-F8","F8-T8","T8-P8","P8-O2",
    "FZ-CZ","CZ-PZ"
]

def feature_family(name):
    if name.startswith("Chi"):
        return "Calibrated WPE"
    elif name.startswith("Delta"):
        return "Temporal Trend"
    elif name.startswith("Volatility"):
        return "Local Volatility"
    elif name == "Synchrony":
        return "Spatial Synchrony"
    return "Other"

def load_global_importance(config_dir):
    path = os.path.join(config_dir, "plots_data", "global_feature_importance.csv")
    df = pd.read_csv(path)
    df["family"] = df["feature"].apply(feature_family)
    df["channel"] = df["feature"].apply(feature_channel)
    return df

def load_patient_importances(config_dir):
    files = sorted(glob.glob(os.path.join(config_dir, "plots_data", "patient_chb*_feature_importance.csv")))
    dfs = []
    for f in files:
        df = pd.read_csv(f)
        pid = int(os.path.basename(f).split("patient_chb")[1].split("_")[0])
        df["patient"] = pid
        df["family"] = df["feature"].apply(feature_family)
        df["channel"] = df["feature"].apply(feature_channel)
        dfs.append(df)
    return pd.concat(dfs, ignore_index=True)

In [ ]:
#load data
df_global = load_global_importance(FINAL_CONFIG)
df_patients = load_patient_importances(FINAL_CONFIG)

#1. TOP 10 global features
top_n = 15
df_top = df_global.sort_values("mean_importance", ascending=False).head(top_n).copy()
palette_family = {
    "Calibrated WPE": "black",
    "Temporal Trend": "#2980b9",
    "Local Volatility": "royalblue",
    "Spatial Synchrony": "mediumblue",
    "Other": "midnightblue"
}
plt.figure(figsize=(10, 6))
sns.barplot(
    data=df_top,
    y="feature",
    x="mean_importance",
    hue="family",
    dodge=False,
    palette=palette_family
)
plt.title("Top 15 Global Feature Importances")
plt.xlabel("Mean Importance")
plt.ylabel("Feature")
plt.legend(title="Feature Family", loc="lower right")
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_OUT, "top15_global_feature_importance.png"), dpi=300, bbox_inches="tight")
plt.show()

#2. level of importance depending on family ß
df_family = (
    df_global.groupby("family", as_index=False)[["mean_importance", "std_importance"]]
    .sum()
    .sort_values("mean_importance", ascending=False)
)

plt.figure(figsize=(8, 5))
sns.barplot(
    data=df_family,
    x="family",
    y="mean_importance",
    palette=[palette_family.get(f, "#7f8c8d") for f in df_family["family"]]
)
plt.title("Aggregated Importance by Feature Family")
plt.xlabel("")
plt.ylabel("Total Mean Importance")
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_OUT, "importance_by_family.png"), dpi=300, bbox_inches="tight")
plt.show()

#3. importance by channel
df_channel = (
    df_global.groupby("channel", as_index=False)["mean_importance"]
    .sum()
    .sort_values("mean_importance", ascending=False)
)
plt.figure(figsize=(10, 6))
sns.barplot(
    data=df_channel,
    y="channel",
    x="mean_importance",
    color="#34495e"
)
plt.title("Aggregated Importance by EEG Channel")
plt.xlabel("Total Mean Importance")
plt.ylabel("Channel")
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_OUT, "importance_by_channel.png"), dpi=300, bbox_inches="tight")
plt.show()

#4. Heatmap
top_heat_n = 15
top_features = df_global.sort_values("mean_importance", ascending=False).head(top_heat_n)["feature"].tolist()
df_heat = df_patients[df_patients["feature"].isin(top_features)].copy()
heatmap_data = df_heat.pivot_table(
    index="feature",
    columns="patient",
    values="importance",
    aggfunc="mean"
).reindex(top_features)
plt.figure(figsize=(12, 6))
sns.heatmap(
    heatmap_data,
    cmap="magma",
    linewidths=0.5,
    linecolor="white"
)
plt.title("Top Feature Importance Across LOPO Test Patients")
plt.xlabel("Held-out Patient")
plt.ylabel("Feature")
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_OUT, "top_features_heatmap_patients.png"), dpi=300, bbox_inches="tight")
plt.show()

#5. Ablation
metric_to_plot = "auc_roc"
rows = []
for config_name, config_dir in CONFIG_MAP.items():
    subject_path = os.path.join(config_dir, "metrics", "subject_metrics.csv")
    dfm = pd.read_csv(subject_path)
    rows.append({
        "Config": config_name,
        "Mean": dfm[metric_to_plot].mean(),
        "Std": dfm[metric_to_plot].std()
    })
df_ablation = pd.DataFrame(rows)
pretty_names = {
    "A_raw_wpe": "A: Raw WPE",
    "B_wpe_calibration": "B: WPE + Calibration",
    "C_full_feature_space": "C: Full Feature Space",
    "D_full_feature_space_persistence": "D: Full + Persistence"
}
df_ablation["ConfigPretty"] = df_ablation["Config"].map(pretty_names)
plt.figure(figsize=(9, 5))
plt.bar(
    df_ablation["ConfigPretty"],
    df_ablation["Mean"],
    yerr=df_ablation["Std"],
    capsize=6,
    color=["#95a5a6", "#3498db", "#2ecc71", "#e67e22"]
)
plt.ylabel("Mean AUC-ROC")
plt.xlabel("")
plt.title("Ablation Study Performance")
plt.ylim(0, max(1.0, df_ablation["Mean"].max() + 0.08))
plt.xticks(rotation=10)
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_OUT, "ablation_aucroc.png"), dpi=300, bbox_inches="tight")
plt.show()

print(f"All plots saved in: {PLOTS_OUT}")

# Inter-patient heterogeneity analysis

In [ ]:
plt.style.use("seaborn-v0_8-whitegrid")

FINAL_CONFIG_DIR = os.path.join(
    ABLATION_ROOT,
    "D_full_feature_space_persistence"
)

SUBJECT_METRICS_PATH = os.path.join(
    FINAL_CONFIG_DIR,
    "metrics",
    "subject_metrics.csv"
)
OUTDIR = os.path.join(FINAL_CONFIG_DIR, "heterogeneity_analysis")
os.makedirs(OUTDIR, exist_ok=True)

#subject metrics
df = pd.read_csv(SUBJECT_METRICS_PATH)
df = df[[
    "patient", "auc_roc", "pr_auc", "accuracy", "precision",
    "sensitivity", "specificity", "f1_score", "far_h"
]].copy()
df = df.sort_values("auc_roc").reset_index(drop=True)

#split following median cut-off
median_auc = df["auc_roc"].median()
df["group"] = np.where(df["auc_roc"] < median_auc, "LOW", "HIGH")
low_patients = df.loc[df["group"] == "LOW", "patient"].astype(int).tolist()
high_patients = df.loc[df["group"] == "HIGH", "patient"].astype(int).tolist()
print("LOW patients:", low_patients)
print("HIGH patients:", high_patients)
df["patient_label"] = df["patient"].apply(lambda x: f"chb{int(x):02d}")

#Summary
print("=" * 60)
print("MEDIAN SPLIT BASED ON FINAL LOPO AUC-ROC")
print("=" * 60)
print(f"Median AUC-ROC = {median_auc:.4f}\n")
print("Group sizes:")
print(df["group"].value_counts(), "\n")
print("Patients ordered by AUC-ROC:")
print(df[["patient_label", "auc_roc", "group"]].to_string(index=False), "\n")
print("LOW group:")
print(df.loc[df["group"] == "LOW", "patient_label"].tolist(), "\n")
print("HIGH group:")
print(df.loc[df["group"] == "HIGH", "patient_label"].tolist(), "\n")
#CSV
group_csv_path = os.path.join(OUTDIR, "patient_groups_median_split.csv")
df.to_csv(group_csv_path, index=False)

In [ ]:
#Descriptive statistics
summary = df.groupby("group")[[
    "auc_roc", "pr_auc", "accuracy", "precision",
    "sensitivity", "specificity", "f1_score", "far_h"
]].agg(["mean", "std", "median", "min", "max"])

summary_path = os.path.join(OUTDIR, "group_summary_stats.csv")
summary.to_csv(summary_path)

print(f"Saved: {summary_path}")

#Patient rank
plt.figure(figsize=(11, 5))
colors = df["group"].map({"LOW": "#c0392b", "HIGH": "#1f618d"})
bars = plt.bar(df["patient_label"], df["auc_roc"], color=colors)
plt.axhline(median_auc, color="black", linestyle="--", linewidth=1.5,
            label=f"Median = {median_auc:.3f}")
plt.ylabel("AUC-ROC")
plt.xlabel("Patient")
plt.title("Patient-wise LOPO AUC-ROC and Median Split")
plt.xticks(rotation=45)
plt.ylim(0, 1.0)
plt.legend()
plt.tight_layout()
plot1_path = os.path.join(OUTDIR, "patient_auc_median_split.png")
plt.savefig(plot1_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved in {plot1_path}")

#Boxplot lov vs high
plt.figure(figsize=(6, 5))
sns.boxplot(data=df, x="group", y="auc_roc", palette={"LOW": "#c0392b", "HIGH": "#1f618d"}, width=0.5)
sns.stripplot(data=df, x="group", y="auc_roc", color="black", alpha=0.65, size=6)
plt.ylabel("AUC-ROC")
plt.xlabel("")
plt.title("Distribution of Patient-wise AUC-ROC by Group")
plt.ylim(0, 1.0)
plt.tight_layout()
plot2_path = os.path.join(OUTDIR, "group_auc_boxplot.png")
plt.savefig(plot2_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved: {plot2_path}")

In [ ]:
FINAL_CONFIG_DIR = os.path.join(ABLATION_ROOT, "D_full_feature_space_persistence")
OUTDIR = os.path.join(FINAL_CONFIG_DIR, "heterogeneity_analysis")
os.makedirs(OUTDIR, exist_ok=True)
subject_metrics_path = os.path.join(FINAL_CONFIG_DIR, "metrics", "subject_metrics.csv")
groups_path = os.path.join(OUTDIR, "patient_groups_median_split.csv")
df_metrics = pd.read_csv(subject_metrics_path)
df_groups = pd.read_csv(groups_path)
metric_cols = [
    "auc_roc", "pr_auc", "accuracy", "precision",
    "sensitivity", "specificity", "f1_score", "far_h"
]
df = df_metrics.merge(df_groups[["patient", "group"]], on="patient", how="inner")


In [ ]:
#Descriptive tables
summary = df.groupby("group")[metric_cols].agg(["mean", "std", "median", "min", "max"])
summary.to_csv(os.path.join(OUTDIR, "low_vs_high_descriptive_stats.csv"))
print("=" * 70)
print("LOW vs HIGH DESCRIPTIVE STATS")
print("=" * 70)
display(summary)

#Non parametric tests
rows = []
for m in metric_cols:
    low_vals = df.loc[df["group"] == "LOW", m].dropna().values
    high_vals = df.loc[df["group"] == "HIGH", m].dropna().values
    stat, pval = mannwhitneyu(low_vals, high_vals, alternative="two-sided")
    rows.append({
        "metric": m,
        "low_mean": np.mean(low_vals),
        "high_mean": np.mean(high_vals),
        "low_median": np.median(low_vals),
        "high_median": np.median(high_vals),
        "U_stat": stat,
        "p_value": pval
    })
df_stats = pd.DataFrame(rows)
df_stats.to_csv(os.path.join(OUTDIR, "low_vs_high_mannwhitney.csv"), index=False)
print("\n" + "=" * 70)
print("LOW vs HIGH - MANN-WHITNEY U TEST")
print("=" * 70)
display(df_stats)
paper_rows = []
for group_name, dfg in df.groupby("group"):
    row = {"group": group_name}
    for m in metric_cols:
        row[m] = f"{dfg[m].mean():.3f} ± {dfg[m].std():.3f}"
    paper_rows.append(row)

df_paper = pd.DataFrame(paper_rows)
df_paper.to_csv(os.path.join(OUTDIR, "low_vs_high_paper_table.csv"), index=False)

print("\n" + "=" * 70)
print("PAPER TABLE")
print("=" * 70)
display(df_paper)

#plots
plot_metrics = ["auc_roc", "sensitivity", "specificity", "far_h"]
pretty_names = {
    "auc_roc": "AUC-ROC",
    "sensitivity": "Sensitivity",
    "specificity": "Specificity",
    "far_h": "FAR/h"
}
df_long = df.melt(
    id_vars=["patient", "group"],
    value_vars=plot_metrics,
    var_name="metric",
    value_name="value"
)
df_long["metric_label"] = df_long["metric"].map(pretty_names)
plt.figure(figsize=(11, 5))
sns.boxplot(
    data=df_long,
    x="metric_label",
    y="value",
    hue="group",
    palette={"LOW": "#c0392b", "HIGH": "#1f618d"},
    width=0.65
)
sns.stripplot(
    data=df_long,
    x="metric_label",
    y="value",
    hue="group",
    dodge=True,
    color="black",
    alpha=0.45,
    size=4
)
handles, labels = plt.gca().get_legend_handles_labels()
plt.legend(handles[:2], labels[:2], title="Group")
plt.xlabel("")
plt.ylabel("Score")
plt.title("LOW vs HIGH patient groups: performance comparison")
plt.tight_layout()
plt.savefig(os.path.join(OUTDIR, "low_vs_high_metrics_boxplot.png"), dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
display(df_paper)

## Heterogeneity_analysis

In [ ]:
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({
    "figure.dpi": 130,
    "axes.titlesize": 15,
    "axes.labelsize": 13,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10
})

#Paths
FINAL_CONFIG_DIR = os.path.join(ABLATION_ROOT, "D_full_feature_space_persistence")
OUTDIR = os.path.join(FINAL_CONFIG_DIR, "heterogeneity_analysis")
os.makedirs(OUTDIR, exist_ok=True)
groups_path = os.path.join(OUTDIR, "patient_groups_median_split.csv")
#Parameters
WARMUP_CUTOFF = 480
FASTWIN = 12
SLOWWIN = 480
#Features names
raw_feature_names = [f"Chi{i+1:02d}" for i in range(18)]
delta_feature_names = [f"Delta{i+1:02d}" for i in range(18)]
vol_feature_names = [f"Volatility{i+1:02d}" for i in range(18)]
sync_feature_names = ["Synchrony"]
full_feature_names = raw_feature_names + delta_feature_names + vol_feature_names + sync_feature_names

#Helpers
def extract_calibrated_wpe(X_patient, patient_id):
    X_raw = extract_raw_wpe(X_patient)
    direction = dir_map.get(int(patient_id), 1)
    X_corrected = X_raw * direction
    return X_corrected.astype(np.float32)

def extract_full_feature_space(X_patient, patient_id):
    X_base = extract_calibrated_wpe(X_patient, patient_id)
    df = pd.DataFrame(X_base)

    ma_fast = df.rolling(window=FASTWIN, min_periods=1).mean()
    ma_slow = df.rolling(window=SLOWWIN, min_periods=1).mean()
    X_delta = (ma_fast - ma_slow).values.astype(np.float32)

    X_vol = (
        df.rolling(window=FASTWIN, min_periods=1)
          .std()
          .fillna(0.0)
          .values
          .astype(np.float32)
    )

    X_sync = (
        df.std(axis=1)
          .rolling(window=FASTWIN, min_periods=1)
          .mean()
          .fillna(0.0)
          .values
          .reshape(-1, 1)
          .astype(np.float32)
    )

    X_full = np.hstack([X_base, X_delta, X_vol, X_sync]).astype(np.float32)
    return X_full

def feature_family(name):
    if name.startswith("Chi"):
        return "Calibrated WPE"
    elif name.startswith("Delta"):
        return "Temporal Trend"
    elif name.startswith("Volatility"):
        return "Local Volatility"
    elif name == "Synchrony":
        return "Spatial Synchrony"
    return "Other"
#Load groups
df_pat = pd.read_csv(groups_path).copy()
df_pat["patient_label"] = df_pat["patient"].apply(lambda x: f"chb{int(x):02d}")
df_pat = df_pat.sort_values("auc_roc").reset_index(drop=True)
print(df_pat[["patient_label", "auc_roc", "group"]].head())
#Compute constrasts
contrast_rows = []

for _, row in df_pat.iterrows():
    pid = int(row["patient"])
    mask = (gall == pid)

    X_patient_raw = Xall[mask]
    y_patient = yall[mask]

    X_patient = extract_full_feature_space(X_patient_raw, pid)

    X_eval = X_patient[WARMUP_CUTOFF:]
    y_eval = y_patient[WARMUP_CUTOFF:]

    if len(np.unique(y_eval)) < 2:
        print(f"Skipping chb{pid:02d}: single class after warm-up")
        continue

    pre_mask = (y_eval == 1)
    inter_mask = (y_eval == 0)

    if pre_mask.sum() == 0 or inter_mask.sum() == 0:
        print(f"Skipping chb{pid:02d}: missing one class after warm-up")
        continue

    pre_mean = X_eval[pre_mask].mean(axis=0)
    inter_mean = X_eval[inter_mask].mean(axis=0)
    contrast = pre_mean - inter_mean

    row_dict = {
        "patient": pid,
        "patient_label": f"chb{pid:02d}",
        "group": row["group"],
        "auc_roc": row["auc_roc"]
    }

    for fname, val in zip(full_feature_names, contrast):
        row_dict[fname] = val

    contrast_rows.append(row_dict)

df_contrast = pd.DataFrame(contrast_rows)
contrast_csv_path = os.path.join(OUTDIR, "patient_feature_contrast_full.csv")
df_contrast.to_csv(contrast_csv_path, index=False)
print("Saved:", contrast_csv_path)
#Heatmap
df_contrast = df_contrast.sort_values("auc_roc").reset_index(drop=True)
heatmap_data = df_contrast[full_feature_names].copy()
heatmap_data.index = [
    f"{p} ({g}, AUC={a:.2f})"
    for p, g, a in zip(df_contrast["patient_label"], df_contrast["group"], df_contrast["auc_roc"])
]
vlim = np.nanpercentile(np.abs(heatmap_data.values), 95)
plt.figure(figsize=(18, 8))
sns.heatmap(
    heatmap_data,
    cmap="coolwarm",
    center=0,
    vmin=-vlim,
    vmax=vlim,
    cbar_kws={"label": "Preictal - Interictal feature contrast"}
)
plt.title("Patient-wise feature contrast heatmap")
plt.xlabel("Features")
plt.ylabel("Patients (ordered by LOPO AUC-ROC)")
plt.tight_layout()
plt.savefig(os.path.join(OUTDIR, "patient_feature_contrast_heatmap.png"), dpi=300, bbox_inches="tight")
plt.show()
#Family level aggregation
family_rows = []
families = ["Calibrated WPE", "Temporal Trend", "Local Volatility", "Spatial Synchrony"]

for _, row in df_contrast.iterrows():
    temp = {
        "patient": row["patient"],
        "patient_label": row["patient_label"],
        "group": row["group"],
        "auc_roc": row["auc_roc"]
    }
    for fam in families:
        fam_feats = [f for f in full_feature_names if feature_family(f) == fam]
        temp[fam] = row[fam_feats].mean()
    family_rows.append(temp)

df_family = pd.DataFrame(family_rows)
family_csv_path = os.path.join(OUTDIR, "patient_feature_contrast_by_family.csv")
df_family.to_csv(family_csv_path, index=False)
print("Saved:", family_csv_path)

family_heatmap = df_family[families].copy()
family_heatmap.index = [
    f"{p} ({g}, AUC={a:.2f})"
    for p, g, a in zip(df_family["patient_label"], df_family["group"], df_family["auc_roc"])
]

vlim_fam = np.nanpercentile(np.abs(family_heatmap.values), 95)

plt.figure(figsize=(8, 8))
sns.heatmap(
    family_heatmap,
    cmap="coolwarm",
    center=0,
    vmin=-vlim_fam,
    vmax=vlim_fam,
    annot=True,
    fmt=".2f",
    cbar_kws={"label": "Mean family contrast"}
)
plt.title("Patient-wise feature contrast by family")
plt.xlabel("Feature families")
plt.ylabel("Patients (ordered by LOPO AUC-ROC)")
plt.tight_layout()
plt.savefig(os.path.join(OUTDIR, "patient_feature_family_contrast_heatmap.png"), dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# root output
if "ABLATION_ROOT" not in globals():
    if "ABLATIONROOT" in globals():
        ABLATION_ROOT = ABLATIONROOT
    else:
        raise NameError("ABLATION_ROOT non definito e ABLATIONROOT non trovato nel notebook.")

# feature extraction
if "extract_full_feature_space" not in globals():
    if "extractfullfeaturespace" in globals():
        extract_full_feature_space = extract_full_feature_space
    else:
        raise NameError("extract_full_feature_space non definita e extractfullfeaturespace non trovato.")

#features names
if "full_feature_space" not in globals():
    if "full_feature_names" in globals():
        full_feature_space = full_feature_names
    else:
        raise NameError("full_feature_space non definito e fullfeaturenames non trovato.")

# undersampling strategy
if "UNDERSAMPLING_STRATEGY" not in globals():
    if "UNDERSAMPLINGSTRATEGY" in globals():
        UNDERSAMPLING_STRATEGY = UNDERSAMPLINGSTRATEGY
    else:
        raise NameError("UNDERSAMPLING_STRATEGY non definita e UNDERSAMPLINGSTRATEGY non trovato.")

#Obtained groups
LOW_PATIENTS  = [23, 18,  9, 14, 22, 10,  4, 15,  1,  6]
HIGH_PATIENTS = [ 5, 13, 21,  8, 17, 16,  7,  3, 20, 12]



In [ ]:
SUBGROUP_ROOT = os.path.join(ABLATION_ROOT, "subgroup_lopo_modelD")
os.makedirs(SUBGROUP_ROOT, exist_ok=True)


def make_local_dirs(root, tag):
    dirs = {
        "root": os.path.join(root, tag),
    }
    dirs["models"] = os.path.join(dirs["root"], "models")
    dirs["metrics"] = os.path.join(dirs["root"], "metrics")
    dirs["predictions"] = os.path.join(dirs["root"], "predictions")
    dirs["plots_data"] = os.path.join(dirs["root"], "plots_data")
    dirs["config"] = os.path.join(dirs["root"], "config_info.json")
    for k, v in dirs.items():
        if k != "config":
            os.makedirs(v, exist_ok=True)
    return dirs


def save_local_config(dirs, payload):
    with open(dirs["config"], "w") as f:
        json.dump(payload, f, indent=2)


def run_lopo_on_subgroup(
    selected_patients,
    subgroup_name,
    config_key="D_full_feature_space_persistence",
    use_persistence=True
):
    selected_patients = np.array(sorted(int(p) for p in selected_patients))
    tag = f"{config_key}_{subgroup_name}"
    dirs = make_local_dirs(SUBGROUP_ROOT, tag)

    save_local_config(dirs, {
        "config_key": config_key,
        "subgroup": subgroup_name,
        "patients": selected_patients.tolist(),
        "validation": "LOPO inside subgroup",
        "n_patients": int(len(selected_patients)),
        "use_persistence": bool(use_persistence),
        "undersampling_strategy": UNDERSAMPLING_STRATEGY,
        "rf_params": RF_PARAMS,
        "blend_alpha": BLEND_ALPHA,
    })

    all_metrics = []
    all_importances = []

    for fold_idx, patient_test in enumerate(selected_patients, start=1):
        test_mask = g_all == patient_test
        X_test_raw = X_all[test_mask]
        y_test_full = y_all[test_mask]

        X_test = extract_full_feature_space(X_test_raw, patient_test)
        univariate_test = extract_univariate_score(X_test_raw, patient_test)

        train_patients = [p for p in selected_patients if p != patient_test]
        X_train_blocks, y_train_blocks, univariate_train_blocks = [], [], []

        for patient_train in train_patients:
            train_mask = g_all == patient_train
            X_train_patient_raw = X_all[train_mask]
            y_train_patient = y_all[train_mask]
            X_train_patient = extract_full_feature_space(X_train_patient_raw, patient_train)
            univariate_train_patient = extract_univariate_score(X_train_patient_raw, patient_train)
            X_train_blocks.append(X_train_patient)
            y_train_blocks.append(y_train_patient)
            univariate_train_blocks.append(univariate_train_patient)

        if len(X_train_blocks) == 0:
            print(f"[{tag}] chb{patient_test:02d} skipped: no training patients.")
            continue

        X_train = np.vstack(X_train_blocks).astype(np.float32)
        y_train = np.concatenate(y_train_blocks).astype(np.int32)
        univariate_train = np.concatenate(univariate_train_blocks).astype(np.float32)

        rus = RandomUnderSampler(sampling_strategy=UNDERSAMPLING_STRATEGY, random_state=42)
        X_res, y_res = rus.fit_resample(X_train, y_train)

        clf = RandomForestClassifier(**RF_PARAMS)
        clf.fit(X_res, y_res)

        logreg_univariate = LogisticRegression(class_weight="balanced", random_state=42)
        logreg_univariate.fit(univariate_train.reshape(-1, 1), y_train)

        model_path = os.path.join(dirs["models"], f"rf_model_chb{patient_test:02d}.joblib")
        joblib.dump(clf, model_path)

        y_prob_rf = clf.predict_proba(X_test)[:, 1]
        y_prob_univariate = logreg_univariate.predict_proba(univariate_test.reshape(-1, 1))[:, 1]
        y_prob_raw = (BLEND_ALPHA * y_prob_rf + (1 - BLEND_ALPHA) * y_prob_univariate).astype(np.float32)
        y_prob_smooth = smooth_probabilities(y_prob_raw, win=SMOOTH_WIN)

        y_eval = y_test_full[WARMUP_CUTOFF:]
        y_score_eval = y_prob_smooth[WARMUP_CUTOFF:]

        if len(np.unique(y_eval)) < 2:
            print(f"[{tag}] chb{patient_test:02d} skipped: single-class after warm-up.")
            continue

        auc_roc = roc_auc_score(y_eval, y_score_eval)
        pr_auc = average_precision_score(y_eval, y_score_eval)

        threshold_opt_nested = choose_threshold_youden_nested(
            train_patients, X_train_blocks, y_train_blocks, univariate_train_blocks,
            RF_PARAMS, BLEND_ALPHA, UNDERSAMPLING_STRATEGY, n_splits=4,
            warmup=WARMUP_CUTOFF, smooth_win=SMOOTH_WIN, random_state=42
        )
        threshold_opt = (
            threshold_opt_nested if threshold_opt_nested is not None
            else choose_threshold_youden(y_eval, y_score_eval)[0]
        )

        y_pred_binary = (y_score_eval >= threshold_opt).astype(int)
        y_pred_final = (
            apply_persistence_rule(y_pred_binary, PERSISTENCE_EPOCHS)
            if use_persistence else y_pred_binary
        )

        epoch_metrics = compute_epoch_metrics(y_eval, y_pred_final)
        event_metrics = compute_event_metrics(y_eval, y_pred_final, epoch_sec=EPOCH_SEC, sph_min=5)

        patient_metrics = {
            "patient": int(patient_test),
            "config": tag,
            "subgroup": subgroup_name,
            "n_epochs_eval": int(len(y_eval)),
            "n_features": int(X_test.shape[1]),
            "auc_roc": auc_roc,
            "pr_auc": pr_auc,
            "threshold": float(threshold_opt),
            **epoch_metrics,
            "far_h": event_metrics["far_h"],
            "false_alarm_events": event_metrics["false_alarm_events"],
            "interictal_hours": event_metrics["interictal_hours"],
            "total_seizures": event_metrics["total_seizures"],
            "predicted_seizures": event_metrics["predicted_seizures"],
            "missed_seizures": event_metrics["missed_seizures"],
            "seizure_sensitivity": event_metrics["seizure_sensitivity"],
            "mean_warning_time_min": event_metrics["mean_warning_time_min"],
            "median_warning_time_min": event_metrics["median_warning_time_min"],
            "far_24h": event_metrics["far_24h"],
            "time_in_warning": event_metrics["time_in_warning"],
        }
        all_metrics.append(patient_metrics)

        if hasattr(clf, "feature_importances_"):
            pd.DataFrame({
                "feature": full_feature_names,
                "importance": clf.feature_importances_
            }).sort_values("importance", ascending=False).to_csv(
                os.path.join(dirs["plots_data"], f"patient_chb{patient_test:02d}_feature_importance.csv"),
                index=False
            )
            all_importances.append(clf.feature_importances_)

        print(
            f"[{tag}] Fold {fold_idx:02d} chb{patient_test:02d} "
            f"AUC={auc_roc:.3f} Sens={patient_metrics['sensitivity']:.3f} "
            f"Spec={patient_metrics['specificity']:.3f} FAR/h={patient_metrics['far_h']:.3f} "
            f"Seizures {event_metrics['predicted_seizures']}/{event_metrics['total_seizures']}"
        )

        del X_train, y_train, X_res, y_res, X_test, y_prob_raw, y_prob_smooth, clf
        gc.collect()

    df_metrics = pd.DataFrame(all_metrics)
    df_metrics.to_csv(os.path.join(dirs["metrics"], "subject_metrics.csv"), index=False)
    df_summary = summarize_metrics(df_metrics, os.path.join(dirs["metrics"], "global_metrics.csv"))

    if len(all_importances) > 0:
        imp_matrix = np.vstack(all_importances)
        pd.DataFrame({
            "feature": full_feature_names,
            "mean_importance": imp_matrix.mean(axis=0),
            "std_importance": imp_matrix.std(axis=0)
        }).sort_values("mean_importance", ascending=False).to_csv(
            os.path.join(dirs["plots_data"], "global_feature_importance.csv"), index=False
        )

    print(f"\nCompleted {tag}")
    display(df_summary)
    return df_metrics, df_summary


#Obtained groups
LOW_PATIENTS  = [23, 18,  9, 14, 22, 10,  4, 15,  1,  6]
HIGH_PATIENTS = [ 5, 13, 21,  8, 17, 16,  7,  3, 20, 12]


In [ ]:
#Execution
df_high_lopo, summary_high_lopo = run_lopo_on_subgroup(HIGH_PATIENTS, "HIGH")
df_low_lopo,  summary_low_lopo  = run_lopo_on_subgroup(LOW_PATIENTS,  "LOW")

## Cross-Group RF

In [ ]:
# 1. train su HIGH, test su LOW
# 2. train su LOW, test su HIGH

CROSSGROUP_ROOT = os.path.join(ABLATION_ROOT, "crossgroup_modelD")
os.makedirs(CROSSGROUP_ROOT, exist_ok=True)


def run_crossgroup_rf(train_patients, test_patients, train_name, test_name,
                       config_key="D_full_feature_space_persistence",
                       use_persistence=True):

    train_patients = np.array(sorted(int(p) for p in train_patients))
    test_patients = np.array(sorted(int(p) for p in test_patients))

    tag = f"{config_key}_train{train_name}_test{test_name}"
    dirs = make_local_dirs(CROSSGROUP_ROOT, tag)

    save_local_config(dirs, {
        "config_key": config_key,
        "tag": tag,
        "validation": "cross-group generalization",
        "train_group": train_name,
        "test_group": test_name,
        "train_patients": train_patients.tolist(),
        "test_patients": test_patients.tolist(),
        "use_persistence": bool(use_persistence),
        "undersampling_strategy": UNDERSAMPLING_STRATEGY,
        "rf_params": RF_PARAMS,
        "blend_alpha": BLEND_ALPHA,
    })

    X_train_blocks, y_train_blocks, univariate_train_blocks = [], [], []
    train_patients_list = train_patients.tolist()

    for patient_train in train_patients_list:
        train_mask = g_all == patient_train
        X_train_patient_raw = X_all[train_mask]
        y_train_patient = y_all[train_mask]
        X_train_patient = extract_full_feature_space(X_train_patient_raw, patient_train)
        univariate_train_patient = extract_univariate_score(X_train_patient_raw, patient_train)
        X_train_blocks.append(X_train_patient)
        y_train_blocks.append(y_train_patient)
        univariate_train_blocks.append(univariate_train_patient)

    if len(X_train_blocks) == 0:
        raise ValueError(f"{tag}: nessun paziente nel training set.")

    X_train = np.vstack(X_train_blocks).astype(np.float32)
    y_train = np.concatenate(y_train_blocks).astype(np.int32)
    univariate_train = np.concatenate(univariate_train_blocks).astype(np.float32)

    rus = RandomUnderSampler(sampling_strategy=UNDERSAMPLING_STRATEGY, random_state=42)
    X_res, y_res = rus.fit_resample(X_train, y_train)

    clf = RandomForestClassifier(**RF_PARAMS)
    clf.fit(X_res, y_res)

    logreg_univariate = LogisticRegression(class_weight="balanced", random_state=42)
    logreg_univariate.fit(univariate_train.reshape(-1, 1), y_train)

    joblib.dump(clf, os.path.join(dirs["models"], f"rf_model_train{train_name}_test{test_name}.joblib"))
    threshold_opt_nested = choose_threshold_youden_nested(
        train_patients_list, X_train_blocks, y_train_blocks, univariate_train_blocks,
        RF_PARAMS, BLEND_ALPHA, UNDERSAMPLING_STRATEGY, n_splits=4,
        warmup=WARMUP_CUTOFF, smooth_win=SMOOTH_WIN, random_state=42
    )

    all_metrics = []

    for fold_idx, patient_test in enumerate(test_patients, start=1):
        test_mask = g_all == patient_test
        X_test_raw = X_all[test_mask]
        y_test_full = y_all[test_mask]

        X_test = extract_full_feature_space(X_test_raw, patient_test)
        univariate_test = extract_univariate_score(X_test_raw, patient_test)

        y_prob_rf = clf.predict_proba(X_test)[:, 1]
        y_prob_univariate = logreg_univariate.predict_proba(univariate_test.reshape(-1, 1))[:, 1]
        y_prob_raw = (BLEND_ALPHA * y_prob_rf + (1 - BLEND_ALPHA) * y_prob_univariate).astype(np.float32)
        y_prob_smooth = smooth_probabilities(y_prob_raw, win=SMOOTH_WIN)

        y_eval = y_test_full[WARMUP_CUTOFF:]
        y_score_eval = y_prob_smooth[WARMUP_CUTOFF:]

        if len(np.unique(y_eval)) < 2:
            print(f"[{tag}] chb{patient_test:02d} skipped: single-class after warm-up.")
            continue

        auc_roc = roc_auc_score(y_eval, y_score_eval)
        pr_auc = average_precision_score(y_eval, y_score_eval)

        threshold_opt = (
            threshold_opt_nested if threshold_opt_nested is not None
            else choose_threshold_youden(y_eval, y_score_eval)[0]
        )

        y_pred_binary = (y_score_eval >= threshold_opt).astype(int)
        y_pred_final = (
            apply_persistence_rule(y_pred_binary, PERSISTENCE_EPOCHS)
            if use_persistence else y_pred_binary
        )

        epoch_metrics = compute_epoch_metrics(y_eval, y_pred_final)
        event_metrics = compute_event_metrics(y_eval, y_pred_final, epoch_sec=EPOCH_SEC, sph_min=5)

        patient_metrics = {
            "patient": int(patient_test),
            "config": tag,
            "train_group": train_name,
            "test_group": test_name,
            "auc_roc": auc_roc,
            "pr_auc": pr_auc,
            "threshold": float(threshold_opt),
            **epoch_metrics,
            "far_h": event_metrics["far_h"],
            "false_alarm_events": event_metrics["false_alarm_events"],
            "interictal_hours": event_metrics["interictal_hours"],
            "total_seizures": event_metrics["total_seizures"],
            "predicted_seizures": event_metrics["predicted_seizures"],
            "missed_seizures": event_metrics["missed_seizures"],
            "seizure_sensitivity": event_metrics["seizure_sensitivity"],
            "mean_warning_time_min": event_metrics["mean_warning_time_min"],
            "median_warning_time_min": event_metrics["median_warning_time_min"],
            "far_24h": event_metrics["far_24h"],
            "time_in_warning": event_metrics["time_in_warning"],
        }
        all_metrics.append(patient_metrics)

        print(
            f"[{tag}] TestFold {fold_idx:02d} chb{patient_test:02d} "
            f"AUC={auc_roc:.3f} Sens={patient_metrics['sensitivity']:.3f} "
            f"Spec={patient_metrics['specificity']:.3f} FAR/h={patient_metrics['far_h']:.3f} "
            f"Seizures {event_metrics['predicted_seizures']}/{event_metrics['total_seizures']}"
        )

        del X_test, y_prob_raw, y_prob_smooth
        gc.collect()

    df_metrics = pd.DataFrame(all_metrics)
    df_metrics.to_csv(os.path.join(dirs["metrics"], "subject_metrics.csv"), index=False)
    df_summary = summarize_metrics(df_metrics, os.path.join(dirs["metrics"], "global_metrics.csv"))

    if hasattr(clf, "feature_importances_"):
        pd.DataFrame({
            "feature": full_feature_names,
            "importance": clf.feature_importances_
        }).sort_values("importance", ascending=False).to_csv(
            os.path.join(dirs["plots_data"], "global_feature_importance.csv"), index=False
        )

    print(f"\nCompleted {tag}")
    display(df_summary)
    return df_metrics, df_summary


In [ ]:
#Execution
df_trainHIGH_testLOW, summary_trainHIGH_testLOW = run_crossgroup_rf(
    train_patients=HIGH_PATIENTS,
    test_patients=LOW_PATIENTS,
    train_name="HIGH",
    test_name="LOW"
)
df_trainLOW_testHIGH, summary_trainLOW_testHIGH = run_crossgroup_rf(
    train_patients=LOW_PATIENTS,
    test_patients=HIGH_PATIENTS,
    train_name="LOW",
    test_name="HIGH"
)

In [ ]:
plt.rcParams.update({
    "figure.dpi": 130,
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10
})
CONFIG_DIR = os.path.join(ABLATION_ROOT, "D_full_feature_space_persistence")
PRED_DIR = os.path.join(CONFIG_DIR, "predictions")
HEAT_OUT = os.path.join(CONFIG_DIR, "wpe_group_analysis")
os.makedirs(HEAT_OUT, exist_ok=True)
subject_metrics_path = os.path.join(CONFIG_DIR, "metrics", "subject_metrics.csv")
df = pd.read_csv(subject_metrics_path)

if "group" not in df.columns:
    median_auc = df["auc_roc"].median()
    df["group"] = np.where(df["auc_roc"] < median_auc, "LOW", "HIGH")

df = df.sort_values("auc_roc").reset_index(drop=True)
df["patient_label"] = df["patient"].apply(lambda x: f"chb{int(x):02d}")

low_patients = df.loc[df["group"] == "LOW", "patient"].tolist()
high_patients = df.loc[df["group"] == "HIGH", "patient"].tolist()
print("LOW:", [f"chb{p:02d}" for p in low_patients])
print("HIGH:", [f"chb{p:02d}" for p in high_patients])
def load_predictions(pid):
    pred_path = os.path.join(PRED_DIR, f"patient_chb{int(pid):02d}_predictions.csv")
    return pd.read_csv(pred_path)

def extract_wpe_signal(X_patient):
    return -np.mean(X_patient[:, :18], axis=1).astype(np.float32)

def normalize_01(x):
    x = np.asarray(x, dtype=np.float32)
    mn, mx = np.nanmin(x), np.nanmax(x)
    if mx - mn < 1e-12:
        return np.zeros_like(x)
    return (x - mn) / (mx - mn)

def safe_quantiles(x):
    return np.nanpercentile(x, [25, 50, 75])

rows = []
patient_cache = {}

for pid in df["patient"].tolist():
    Xp = Xall[gall == pid]
    yp = yall[gall == pid]
    pred = load_predictions(pid)
    wpe = extract_wpe_signal(Xp)
    wpe_smooth = pd.Series(wpe).rolling(window=150, center=True, min_periods=1).mean().values
    ytrue = pred["y_true_full"].values
    iseval = pred["is_eval_epoch"].values if "is_eval_epoch" in pred.columns else pred["isevalepoch"].values
    if "y_prob_smooth_full" in pred.columns:
        yprob = pred["y_prob_smooth_full"].values
    else:
        yprob = pred["yprobsmoothfull"].values
    evalmask = np.asarray(iseval) == 1
    preictalmask = (ytrue == 1)
    interictalmask = (ytrue == 0)
    rows.append({
        "patient": pid,
        "patient_label": f"chb{int(pid):02d}",
        "group": df.loc[df["patient"] == pid, "group"].iloc[0],
        "auc_roc": df.loc[df["patient"] == pid, "auc_roc"].iloc[0],
        "wpe_mean": float(np.mean(wpe_smooth)),
        "wpe_std": float(np.std(wpe_smooth)),
        "wpe_iqr": float(np.percentile(wpe_smooth, 75) - np.percentile(wpe_smooth, 25)),
        "wpe_preictal_mean": float(np.mean(wpe_smooth[preictalmask])) if np.any(preictalmask) else np.nan,
        "wpe_interictal_mean": float(np.mean(wpe_smooth[interictalmask])) if np.any(interictalmask) else np.nan,
        "wpe_preictal_minus_interictal": float(np.mean(wpe_smooth[preictalmask]) - np.mean(wpe_smooth[interictalmask])) if np.any(preictalmask) and np.any(interictalmask) else np.nan,
        "wpe_eval_std": float(np.std(wpe_smooth[evalmask])) if np.any(evalmask) else np.nan
    })
    patient_cache[pid] = {
        "wpe": wpe,
        "wpe_smooth": wpe_smooth,
        "ytrue": ytrue,
        "yprob": yprob,
        "evalmask": evalmask
    }

df_wpe = pd.DataFrame(rows).sort_values("auc_roc").reset_index(drop=True)
df_wpe.to_csv(os.path.join(HEAT_OUT, "patient_wpe_summary.csv"), index=False)

#Violin plot
plt.figure(figsize=(7, 5))
sns.violinplot(
    data=df_wpe,
    x="group",
    y="wpe_mean",
    palette={"LOW": "midnightblue", "HIGH": "royalblue"},
    inner="box",
    cut=0
)
sns.stripplot(
    data=df_wpe,
    x="group",
    y="wpe_mean",
    color="black",
    alpha=0.6,
    size=5
)
plt.xlabel("")
plt.ylabel("Mean WPE (smoothed)")
plt.title("WPE distribution by group")
plt.tight_layout()
plt.savefig(os.path.join(HEAT_OUT, "wpe_violin_low_high.png"), dpi=300, bbox_inches="tight")
plt.show()

#Heatmap
all_series = []
for pid in df_wpe["patient"].tolist():
    s = patient_cache[pid]["wpe_smooth"]
    s = pd.Series(normalize_01(s), name=f"chb{int(pid):02d}")
    all_series.append(s.reset_index(drop=True))

max_len = max(len(s) for s in all_series)
mat = []
for s in all_series:
    arr = s.values
    if len(arr) < max_len:
        arr = np.pad(arr, (0, max_len - len(arr)), constant_values=np.nan)
    mat.append(arr)

heat_df = pd.DataFrame(mat, index=df_wpe["patient_label"].tolist())
heat_df.to_csv(os.path.join(HEAT_OUT, "wpe_heatmap_matrix.csv"))

plt.figure(figsize=(14, 6))
sns.heatmap(
    heat_df,
    cmap="magma",
    cbar_kws={"label": "Normalized WPE"},
    yticklabels=True
)
plt.xlabel("Time")
plt.ylabel("Patient")
plt.title("WPE heatmap across patients ordered by AUC")
plt.tight_layout()
plt.savefig(os.path.join(HEAT_OUT, "wpe_heatmap_patients.png"), dpi=300, bbox_inches="tight")
plt.show()

#PREICTAL vs INTERICTAL SCATTER / DENSITY
df_scatter_rows = []
for pid in df_wpe["patient"].tolist():
    wpe_s = patient_cache[pid]["wpe_smooth"]
    ytrue = patient_cache[pid]["ytrue"]
    evalmask = patient_cache[pid]["evalmask"]
    pre = wpe_s[(ytrue == 1) & evalmask]
    inter = wpe_s[(ytrue == 0) & evalmask]
    if len(pre) == 0 or len(inter) == 0:
        continue
    pre_q = safe_quantiles(pre)
    inter_q = safe_quantiles(inter)
    df_scatter_rows.append({
        "patient": pid,
        "patient_label": f"chb{int(pid):02d}",
        "group": df_wpe.loc[df_wpe["patient"] == pid, "group"].iloc[0],
        "auc_roc": df_wpe.loc[df_wpe["patient"] == pid, "auc_roc"].iloc[0],
        "pre_median": pre_q[1],
        "inter_median": inter_q[1],
        "delta_median": pre_q[1] - inter_q[1],
        "pre_iqr": pre_q[2] - pre_q[0],
        "inter_iqr": inter_q[2] - inter_q[0]
    })
df_scatter = pd.DataFrame(df_scatter_rows)
df_scatter.to_csv(os.path.join(HEAT_OUT, "wpe_preictal_interictal_summary.csv"), index=False)
plt.figure(figsize=(7, 6))
sns.scatterplot(
    data=df_scatter,
    x="inter_median",
    y="pre_median",
    hue="group",
    style="group",
    palette={"LOW": "#c0392b", "HIGH": "#1f618d"},
    s=90
)
mx = max(df_scatter["pre_median"].max(), df_scatter["inter_median"].max())
mn = min(df_scatter["pre_median"].min(), df_scatter["inter_median"].min())
plt.plot([mn, mx], [mn, mx], "--", color="gray", lw=1)
plt.xlabel("Interictal WPE median")
plt.ylabel("Preictal WPE median")
plt.title("Preictal vs Interictal WPE separation")
plt.tight_layout()
plt.savefig(os.path.join(HEAT_OUT, "wpe_preictal_interictal_scatter.png"), dpi=300, bbox_inches="tight")
plt.show()

#Time series of representative patients
def pick_representatives(df_in, group_name, n=2):
    sub = df_in[df_in["group"] == group_name].sort_values("auc_roc")
    if len(sub) == 0:
        return []
    if len(sub) <= n:
        return sub["patient"].tolist()
    return [sub.iloc[0]["patient"], sub.iloc[-1]["patient"]] if n == 2 else sub["patient"].tolist()[:n]

rep_low = pick_representatives(df_wpe, "LOW", n=2)
rep_high = pick_representatives(df_wpe, "HIGH", n=2)
fig, axes = plt.subplots(2, 2, figsize=(15, 8), sharex=False, sharey=False)
rep_list = [("HIGH", rep_high), ("LOW", rep_low)]
for r, (gname, pids) in enumerate(rep_list):
    for c, pid in enumerate(pids):
        ax = axes[r, c]
        wpe_s = patient_cache[pid]["wpe_smooth"]
        ytrue = patient_cache[pid]["ytrue"]
        evalmask = patient_cache[pid]["evalmask"]
        t = np.arange(len(wpe_s)) * 2.5 / 60.0
        ax.plot(t, wpe_s, color="#2c3e50", lw=1.6, label="WPE")
        if np.any(ytrue == 1):
            ax.fill_between(t, np.nanmin(wpe_s), np.nanmax(wpe_s), where=(ytrue == 1), color="#e74c3c", alpha=0.15, label="Preictal")
        ax.axvline(np.argmax(evalmask) * 2.5 / 60.0 if np.any(evalmask) else 0, color="gray", ls="--", lw=1, alpha=0.6)
        ax.set_title(f"{gname} - chb{int(pid):02d} (AUC {df_wpe.loc[df_wpe['patient']==pid, 'auc_roc'].iloc[0]:.3f})")
        ax.set_xlabel("Time (min)")
        ax.set_ylabel("WPE")
        ax.grid(alpha=0.2)
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=2, frameon=False)
plt.suptitle("Representative WPE time series: HIGH vs LOW", y=1.02, fontsize=15)
plt.tight_layout()
plt.savefig(os.path.join(HEAT_OUT, "wpe_representative_timeseries.png"), dpi=300, bbox_inches="tight")
plt.show()
print("Saved outputs in:", HEAT_OUT)